# Deep Learning PR 2 — Adult Income Classification

## Project Overview
Build and evaluate Multilayer Perceptron (MLP) models to predict whether an adult's annual income is above $50K using the Adult Census Income dataset.

## Objective
Compare activation functions, weight initializers, loss functions, Batch Normalization and optimizers using a reproducible binary-classification workflow.

**Target:** `income` (`<=50K` → 0, `>50K` → 1). The notebook follows all seven task groups in the supplied project definition. It uses a stratified hold-out test set and keeps preprocessing fit operations inside the training data to prevent leakage.

> **Before running:** Place the merged `adult.csv` file beside this notebook. The expected file has 48,842 rows and 15 columns. Download it from the [UCI Adult dataset](https://archive.ics.uci.edu/dataset/2/adult) or the [Kaggle CSV](https://www.kaggle.com/datasets/wenruliu/adult-income-dataset). The UCI/Kaggle source file is not included in this project bundle.

## 1. Libraries and Reproducibility
**What:** Import the tools used for data handling, plots, preprocessing and neural networks.  
**Why:** Consistent imports and fixed random seeds make experiments easier to reproduce.  
**Example:** Like using the same measuring tools for every science experiment.  
**Intuition / expected output:** The cell imports libraries, fixes random seeds and creates the `plots/` folder.

In [ ]:
import os, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report, confusion_matrix,
                             roc_curve)

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
PLOT_DIR = Path("plots"); PLOT_DIR.mkdir(exist_ok=True)
EPOCHS = 50
BATCH_SIZE = 64
sns.set_theme(style="whitegrid", palette="deep")
print("TensorFlow:", tf.__version__)
print("Plot directory:", PLOT_DIR.resolve())

## 2. Dataset Description
The Adult Income dataset is based on 1994 US Census records. It contains demographic and employment attributes such as age, workclass, education level, occupation, weekly working hours and country of origin. The target indicates whether income is above $50K.

**Why this dataset?** It combines numeric and categorical columns and has an imbalanced target, making it suitable for studying preprocessing and ANN behaviour on tabular data.

**Expected output:** Dataset dimensions, data types and sample records. The project specification expects 48,842 rows × 15 columns before cleaning.

In [ ]:
DATA_PATH = Path("adult.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "adult.csv was not found. Download the merged Adult Income CSV from the UCI/Kaggle "
        "source listed in the README and place it in the same folder as this notebook."
    )
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
print("\nData types:")
display(df.dtypes.to_frame("dtype"))

## 3. Task 1 — Data Loading, Cleaning & Exploratory Data Analysis
### 3.1 Clean text and missing values
**What:** Remove whitespace and convert the raw `?` marker into a real missing value.  
**Why:** Whitespace can create duplicate categories; `?` must be treated as missing, not as a valid category.  
**Real-life example:** A blank field in a paper form is not a person's job title.  
**Intuition / expected output:** Missing-value counts should mainly appear in `workclass`, `occupation` and `native.country`.

**Missing-value decision:** The assignment asks us to drop rows with any missing values. This is followed here. These are categorical fields with multiple categories; mode imputation could over-represent the most common category. The number of removed rows is printed rather than assumed.

In [ ]:
df.columns = df.columns.str.strip()
object_columns = df.select_dtypes(include="object").columns
for column in object_columns:
    df[column] = df[column].str.strip()
df.replace("?", np.nan, inplace=True)
print("Missing values before dropping:")
display(df.isna().sum().to_frame("missing_count"))
rows_before = len(df)
df.dropna(inplace=True)
print(f"Rows removed for missing values: {rows_before - len(df):,}")
duplicate_count = int(df.duplicated().sum())
print("Exact duplicate rows found:", duplicate_count)
if duplicate_count:
    df.drop_duplicates(inplace=True)
    print(f"Exact duplicate rows removed: {duplicate_count:,}")
print("Shape after missing-value and duplicate handling:", df.shape)

### 3.2 Remove redundant / non-predictive columns and encode target
**What:** Drop `fnlwgt` and the redundant text `education` field; retain `education.num`. Convert the target to 0/1.  
**Why:** `fnlwgt` is a census sampling weight, not a predictive demographic feature; `education.num` already represents education numerically.  
**Example:** Keep one clean numeric representation instead of storing the same education level twice.  
**Expected output:** A cleaned frame with approximately 46.4K rows and 13 columns, depending on the exact CSV version.

In [ ]:
required_drop = [c for c in ["fnlwgt", "education"] if c in df.columns]
df.drop(columns=required_drop, inplace=True)
if "income" not in df.columns:
    raise ValueError("Target column 'income' is missing. Check that the correct Adult CSV is loaded.")
df["income"] = df["income"].astype(str).str.strip().str.rstrip(".")
df["income"] = (df["income"] == ">50K").astype("int32")
print("Cleaned shape:", df.shape)
print("Target counts:")
display(df["income"].value_counts().sort_index().rename(index={0:"<=50K (0)",1:">50K (1)"}).to_frame("count"))
print("Target proportions:")
display(df["income"].value_counts(normalize=True).sort_index().to_frame("proportion"))

### 3.3 Target class balance
**What:** Plot the number of examples in each income class.  
**Why:** The dataset is approximately 75% class 0 and 25% class 1. A classifier that always predicts class 0 could reach about 75% accuracy while completely missing high-income cases.  
**Real-life example:** In fraud detection, a model that says “not fraud” for everyone may look accurate because fraud is rare, but it is not useful.  
**Expected output:** A count plot with annotated counts. We therefore track precision, recall and F1 for class 1 as well as accuracy.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
order = [0, 1]
sns.countplot(data=df, x="income", order=order, ax=ax, color="#8CB9D9")
ax.set(title="Adult Income — Class Balance", xlabel="Income class (0: <=50K, 1: >50K)", ylabel="Records")
for bar in ax.patches:
    ax.annotate(f"{int(bar.get_height()):,}", (bar.get_x()+bar.get_width()/2, bar.get_height()),
                ha="center", va="bottom", xytext=(0, 3), textcoords="offset points")
fig.tight_layout(); fig.savefig(PLOT_DIR/"eda_class_balance.png", dpi=160, bbox_inches="tight"); plt.show()

### 3.4 Exploratory visualisations
Each chart answers a different question: age distribution, working hours, education level and numeric-feature relationships. Interpretations must be based on the generated plots for this particular CSV; do not assume a pattern before viewing the output.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(data=df, x="age", hue="income", bins=30, stat="density", common_norm=False,
             element="step", ax=ax)
ax.set(title="Age Distribution by Income Class", xlabel="Age", ylabel="Density")
fig.tight_layout(); fig.savefig(PLOT_DIR/"eda_age_by_income.png", dpi=160, bbox_inches="tight"); plt.show()
print("Interpretation prompt: compare the age distributions and note where the >50K group is concentrated.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x="income", y="hours.per.week", ax=ax, color="#A9C9E8")
ax.set(title="Weekly Working Hours by Income Class", xlabel="Income class (0: <=50K, 1: >50K)", ylabel="Hours per week")
fig.tight_layout(); fig.savefig(PLOT_DIR/"eda_hours_by_income.png", dpi=160, bbox_inches="tight"); plt.show()
print("Interpretation prompt: compare medians, spread and outliers between the two classes.")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.countplot(data=df, x="education.num", hue="income", ax=ax)
ax.set(title="Education Level by Income Class", xlabel="Education number", ylabel="Records")
ax.legend(title="Income", labels=["<=50K (0)", ">50K (1)"])
fig.tight_layout(); fig.savefig(PLOT_DIR/"eda_education_by_income.png", dpi=160, bbox_inches="tight"); plt.show()
print("Interpretation prompt: inspect how the proportion of >50K cases changes across education levels.")

### 3.5 Correlation heatmap
**What:** Display correlations among numeric columns after target encoding.  
**Why:** Correlation gives a quick view of linear relationships; it does not prove causation and may miss nonlinear patterns.  
**Example:** A positive correlation between education number and income means higher education numbers tend to co-occur with class 1 in this sample.  
**Expected output:** Annotated numeric correlation heatmap.

In [ ]:
numeric_for_corr = df.select_dtypes(include=np.number)
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(numeric_for_corr.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Heatmap — Numeric Features and Target")
fig.tight_layout(); fig.savefig(PLOT_DIR/"eda_correlation_heatmap.png", dpi=160, bbox_inches="tight"); plt.show()

### 3.6 Train / validation / test split, One-Hot Encoding and scaling
**What:** Split the raw cleaned features first, then fit encoders/scalers using training data only.  
**Why:** Fitting preprocessing on the full dataset leaks information from validation/test into training. This notebook keeps the required `pd.get_dummies(drop_first=True)` and numeric-only `StandardScaler`, but applies them in a leakage-safe order.  
**Real-life example:** Do not use the exam answer sheet to prepare the study notes.  
**Expected output:** Aligned train/validation/test matrices, with the same encoded columns and preserved class ratios. Binary one-hot columns remain 0/1 and are not scaled.

In [ ]:
TARGET = "income"
NUMERIC_COLUMNS = ["age", "education.num", "capital.gain", "capital.loss", "hours.per.week"]
X_raw = df.drop(columns=[TARGET]).copy()
y = df[TARGET].astype("int32").copy()
X_train_full, X_test_raw, y_train_full, y_test = train_test_split(
    X_raw, y, test_size=0.20, random_state=SEED, stratify=y
)
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.10, random_state=SEED, stratify=y_train_full
)
# One-hot encode each partition separately, then align columns to training schema.
X_train = pd.get_dummies(X_train_raw, drop_first=True, dtype=np.float32)
X_val = pd.get_dummies(X_val_raw, drop_first=True, dtype=np.float32).reindex(columns=X_train.columns, fill_value=0)
X_test = pd.get_dummies(X_test_raw, drop_first=True, dtype=np.float32).reindex(columns=X_train.columns, fill_value=0)
scaler = StandardScaler()
scale_columns = [c for c in NUMERIC_COLUMNS if c in X_train.columns]
X_train.loc[:, scale_columns] = scaler.fit_transform(X_train[scale_columns]).astype(np.float32)
X_val.loc[:, scale_columns] = scaler.transform(X_val[scale_columns]).astype(np.float32)
X_test.loc[:, scale_columns] = scaler.transform(X_test[scale_columns]).astype(np.float32)
# Keras consumes dense float32 arrays; preserve the column order for all partitions.
X_train = X_train.astype(np.float32); X_val = X_val.astype(np.float32); X_test = X_test.astype(np.float32)
y_train = y_train.astype(np.float32); y_val = y_val.astype(np.float32); y_test = y_test.astype(np.int32)
print("Encoded feature count:", X_train.shape[1])
print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)
print("Train class ratio:\n", y_train.value_counts(normalize=True).sort_index())
print("Validation class ratio:\n", y_val.value_counts(normalize=True).sort_index())
print("Test class ratio:\n", y_test.value_counts(normalize=True).sort_index())
assert list(X_train.columns) == list(X_val.columns) == list(X_test.columns)
assert not X_train.isna().any().any() and not X_val.isna().any().any() and not X_test.isna().any().any()

## 4. Shared Experiment Utilities
**What:** Define reusable functions for model construction, training, validation metrics and test evaluation.  
**Why:** Every experiment should change only the factor being studied. A shared builder reduces accidental differences between models.  
**Intuition:** One standard recipe, with one ingredient changed at a time.  
**Expected output:** A reusable `build_ann()` function and metric helpers.

In [ ]:
def make_optimizer(optimizer="adam"):
    """Return a fresh optimizer instance for each model."""
    if isinstance(optimizer, str):
        key = optimizer.lower()
        if key == "sgd": return keras.optimizers.SGD(learning_rate=0.01)
        if key == "rmsprop": return keras.optimizers.RMSprop(learning_rate=0.001)
        if key == "adam": return keras.optimizers.Adam(learning_rate=0.001)
        raise ValueError(f"Unsupported optimizer: {optimizer}")
    return optimizer


def build_ann(input_dim, hidden_units=(128, 64), activation="relu",
              initializer="glorot_uniform", use_batch_norm=False,
              optimizer="adam", loss="binary_crossentropy"):
    """Build a two-hidden-layer MLP; BatchNorm is placed before activation."""
    model = keras.Sequential(name="adult_income_mlp")
    model.add(keras.layers.Input(shape=(input_dim,)))
    for units in hidden_units:
        model.add(keras.layers.Dense(units, kernel_initializer=initializer, use_bias=not use_batch_norm))
        if use_batch_norm:
            model.add(keras.layers.BatchNormalization())
        model.add(keras.layers.Activation(activation))
    model.add(keras.layers.Dense(1, activation="sigmoid", name="income_probability"))
    model.compile(optimizer=make_optimizer(optimizer), loss=loss, metrics=["accuracy"])
    return model


def class1_metrics(y_true, probabilities, threshold=0.5):
    predictions = (np.asarray(probabilities).reshape(-1) >= threshold).astype(int)
    return {
        "Test Acc": accuracy_score(y_true, predictions),
        "Prec(1)": precision_score(y_true, predictions, pos_label=1, zero_division=0),
        "Recall(1)": recall_score(y_true, predictions, pos_label=1, zero_division=0),
        "F1(1)": f1_score(y_true, predictions, pos_label=1, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, np.asarray(probabilities).reshape(-1)),
    }


def evaluate_model(model, X_eval=X_test, y_eval=y_test, verbose=True):
    probs = model.predict(X_eval, verbose=0).ravel()
    metrics = class1_metrics(y_eval, probs)
    if verbose:
        print(pd.Series(metrics).round(4))
    return metrics, probs


def fit_model(model, epochs=EPOCHS, class_weight=None, verbose=0):
    return model.fit(X_train, y_train, validation_data=(X_val, y_val),
                     epochs=epochs, batch_size=BATCH_SIZE, class_weight=class_weight,
                     verbose=verbose)


def save_figure(filename):
    plt.tight_layout()
    plt.savefig(PLOT_DIR / filename, dpi=160, bbox_inches="tight")
    plt.show()

results = []
model_registry = {}
probability_registry = {}

def add_result(name, model, history, activation="relu", initializer="glorot_uniform",
               loss_name="binary_crossentropy", batch_norm=False, optimizer="adam"):
    metrics, probs = evaluate_model(model, verbose=False)
    row = {"Model": name, "Activation": activation, "Init": initializer,
           "Loss": loss_name, "BatchNorm": batch_norm, "Optimizer": str(optimizer)}
    row.update(metrics)
    results.append(row); model_registry[name] = model; probability_registry[name] = probs
    return metrics, probs

print("Model builder ready. Input features:", X_train.shape[1])

## 5. Task 2 — MLP Architecture: Baseline ANN
### 5.1 Architecture and parameter count
**What:** Build a network with Dense(128, ReLU) → Dense(64, ReLU) → Dense(1, sigmoid).  
**Why:** A fully connected MLP is a natural baseline for structured numeric and one-hot encoded features. Hidden layers can learn nonlinear combinations such as age × education.  
**Why sigmoid output, not softmax?** One sigmoid neuron returns the probability of the positive class in (0, 1). For two classes, this is simpler than using two softmax output neurons.  
**Expected output:** Model summary and a manually calculated parameter count.

In [ ]:
input_dim = X_train.shape[1]
model_base = build_ann(input_dim=input_dim)
model_base.summary()
manual_params = (input_dim * 128 + 128) + (128 * 64 + 64) + (64 * 1 + 1)
print("Manual parameter count:", manual_params)
print("Keras parameter count:", model_base.count_params())
assert manual_params == model_base.count_params()

### 5.2 Train the baseline
**What:** Train for 50 epochs with batch size 64 and monitor the validation set.  
**Why:** Training curves help detect underfitting and overfitting.  
**Example:** Compare practice-test performance with held-out mock-test performance.  
**Expected output:** Loss and accuracy curves for training and validation.

In [ ]:
history_base = fit_model(model_base, epochs=50, verbose=1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history_base.history["loss"], label="Train"); axes[0].plot(history_base.history["val_loss"], label="Validation")
axes[0].set(title="Loss", xlabel="Epoch", ylabel="Binary cross-entropy"); axes[0].legend()
axes[1].plot(history_base.history["accuracy"], label="Train"); axes[1].plot(history_base.history["val_accuracy"], label="Validation")
axes[1].set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy"); axes[1].legend()
fig.suptitle("Baseline ANN — ReLU · Glorot Uniform · Adam · BCE")
save_figure("baseline_training_curves.png")

### 5.3 Baseline test evaluation
**What:** Evaluate predictions on the untouched test set.  
**Why:** The test set estimates performance on unseen records. The minority-class metrics are especially important because class 1 is less common.  
**Expected output:** Classification report, confusion matrix and recorded class-1 precision, recall and F1.

In [ ]:
baseline_metrics, baseline_probs = add_result("Baseline", model_base, history_base)
baseline_pred = (baseline_probs >= 0.5).astype(int)
print(classification_report(y_test, baseline_pred, target_names=["<=50K", ">50K"], zero_division=0))
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test, baseline_pred), annot=True, fmt="d", cmap="Blues",
            xticklabels=["<=50K", ">50K"], yticklabels=["<=50K", ">50K"], ax=ax)
ax.set(title="Baseline ANN — Confusion Matrix", xlabel="Predicted", ylabel="Actual")
save_figure("baseline_confusion_matrix.png")

**Interpretation:** Read the class-1 row of the classification report. Precision answers “of the people predicted >50K, how many really are >50K?” Recall answers “of all actual >50K people, how many did we find?” F1 balances these two measures. Tabular data suits an MLP because each input neuron represents one scaled numeric or encoded category feature, while Dense layers learn weighted combinations and nonlinear interactions.

## 6. Task 3 — Activation Functions: Comparison Experiment
### 6.1 Train ReLU, Tanh, Sigmoid and ELU
**What:** Change only the hidden-layer activation.  
**Why:** Activation functions add nonlinearity and influence gradient flow.  
**Example:** Different activation functions are like different response curves applied to the same input signal.  
**Expected output:** Four 50-epoch histories and a comparison of validation accuracy / class-1 F1.

In [ ]:
activation_models, activation_histories, activation_metrics = {}, {}, {}
activation_test_metrics = {}
for activation_name in ["relu", "tanh", "sigmoid", "elu"]:
    print("Training activation:", activation_name)
    model = build_ann(input_dim, activation=activation_name)
    hist = fit_model(model, epochs=50)
    val_probs = model.predict(X_val, verbose=0).ravel()
    val_metrics = class1_metrics(y_val, val_probs)
    activation_models[activation_name] = model
    activation_histories[activation_name] = hist
    activation_metrics[activation_name] = val_metrics
    activation_test_metrics[activation_name], _ = add_result(
        f"Activation - {activation_name}", model, hist, activation=activation_name
    )
history_relu = activation_histories["relu"]
history_tanh = activation_histories["tanh"]
history_sigmoid = activation_histories["sigmoid"]
history_elu = activation_histories["elu"]
model_relu = activation_models["relu"]; model_sigmoid = activation_models["sigmoid"]

### 6.2 Activation comparison plot
**What:** Compare training and validation accuracy in a 2×2 layout.  
**Why:** The 75% horizontal reference is the approximate majority-class baseline; validation F1 is also shown in each title.  
**Expected output:** Four panels, one for each activation, with train/validation curves.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (name, hist) in zip(axes.flat, activation_histories.items()):
    ax.plot(hist.history["accuracy"], color="#CC0000", label="Train")
    ax.plot(hist.history["val_accuracy"], color="#2C2C2C", label="Validation")
    ax.axhline(0.75, color="gray", linestyle="--", label="75% reference")
    m = activation_metrics[name]
    ax.set(title=f"{name.title()} | Val Acc={hist.history['val_accuracy'][-1]:.3f} | Val F1(1)={m['F1(1)']:.3f}",
           xlabel="Epoch", ylabel="Accuracy")
    ax.legend(fontsize=8)
fig.suptitle("Activation Function Comparison — Adult Income")
save_figure("activation_comparison.png")

### 6.3 ReLU dead-neuron diagnostic
**What:** Inspect first hidden-layer activations on 500 test examples.  
**Why:** ReLU outputs exactly zero for negative pre-activation values. Neurons that remain inactive can stop contributing.  
**Example:** A light switch that never turns on cannot help the room.  
**Expected output:** Fraction of zero activations and histogram of each neuron's mean activation. ELU can reduce this risk because it permits small negative outputs for negative inputs: α(eˣ − 1).

In [ ]:
relu_hidden = keras.Model(inputs=model_relu.inputs, outputs=model_relu.layers[1].output)
# With BN disabled, layers are Input, Dense, Activation, Dense, Activation, Output.
relu_activations = relu_hidden.predict(X_test[:500], verbose=0)
dead_fraction = float(np.mean(relu_activations == 0))
mean_activation_per_neuron = relu_activations.mean(axis=0)
print(f"Dead activation fraction (zero outputs): {dead_fraction:.4f}")
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(mean_activation_per_neuron, bins=25, edgecolor="black")
ax.set(title="ReLU First-Layer Activation Distribution", xlabel="Mean activation per neuron", ylabel="Neuron count")
save_figure("relu_activation_distribution.png")

### 6.4 Sigmoid gradient-flow diagnostic
**What:** Use `tf.GradientTape` to measure mean absolute gradient magnitude for each Dense layer.  
**Why:** Sigmoid derivatives are at most 0.25. Multiplying small derivatives through layers can shrink early-layer gradients (vanishing gradients).  
**Expected output:** A horizontal bar chart for the two hidden Dense layers and output layer. A much smaller first-layer gradient is evidence consistent with vanishing gradients.

In [ ]:
x_batch = tf.convert_to_tensor(X_train.iloc[:BATCH_SIZE].to_numpy(), dtype=tf.float32)
y_batch = tf.convert_to_tensor(y_train.iloc[:BATCH_SIZE].to_numpy().reshape(-1, 1), dtype=tf.float32)
with tf.GradientTape() as tape:
    p = model_sigmoid(x_batch, training=True)
    loss_value = tf.reduce_mean(keras.losses.binary_crossentropy(y_batch, p))
grads = tape.gradient(loss_value, model_sigmoid.trainable_variables)
# Kernel and bias gradients occur in pairs for each Dense layer.
gradient_by_layer = []
for i in range(0, len(grads), 2):
    layer_grads = [g for g in grads[i:i+2] if g is not None]
    gradient_by_layer.append(float(tf.reduce_mean([tf.reduce_mean(tf.abs(g)) for g in layer_grads])))
layer_names = ["Layer 1", "Layer 2", "Output"][:len(gradient_by_layer)]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(layer_names, gradient_by_layer, color="#8CB9D9")
ax.set(title="Gradient Magnitude by Layer — Sigmoid Hidden Activations", xlabel="Mean |gradient|")
save_figure("sigmoid_gradient_magnitudes.png")
display(pd.DataFrame({"Layer":layer_names,"Mean absolute gradient":gradient_by_layer}))

### 6.5 Activation summary and recommendation
The table below combines standard theoretical properties with the measured validation accuracy and test F1 from this run. Sigmoid remains the output activation for binary classification regardless of the hidden activation selected.

In [ ]:
activation_summary = pd.DataFrame([
    ["ReLU", "max(0,x)", "[0,∞)", "No", "Low–moderate", "Possible", activation_histories["relu"].history["val_accuracy"][-1], activation_metrics["relu"]["F1(1)"]],
    ["Tanh", "tanh(x)", "[-1,1]", "Yes", "High in saturation", "No", activation_histories["tanh"].history["val_accuracy"][-1], activation_metrics["tanh"]["F1(1)"]],
    ["Sigmoid (hidden)", "1/(1+e^-x)", "(0,1)", "No", "High", "No", activation_histories["sigmoid"].history["val_accuracy"][-1], activation_metrics["sigmoid"]["F1(1)"]],
    ["ELU", "x if x>0 else α(eˣ−1)", "(-α,∞)", "Approximately", "Lower than sigmoid", "Lower risk", activation_histories["elu"].history["val_accuracy"][-1], activation_metrics["elu"]["F1(1)"]],
], columns=["Activation","Formula","Output range","Zero-centred","Vanishing-gradient risk","Dead-neuron risk","Final val accuracy","Validation F1(1)"])
display(activation_summary.round(4))
best_activation = max(activation_metrics, key=lambda k: activation_metrics[k]["F1(1)"])
print("Best activation by held-out class-1 F1 in this run:", best_activation)

## 7. Task 4 — Weight Initialization Techniques
### 7.1 Why initialization matters
All-zero weights create a symmetry problem: neurons in the same layer receive identical updates and remain indistinguishable. Glorot/Xavier uses variance approximately 2/(fan_in + fan_out), useful for tanh/sigmoid-style networks. He uses variance approximately 2/fan_in to account for ReLU setting negative activations to zero.

**What:** Compare Glorot Uniform, Glorot Normal, He Uniform, He Normal and Zeros.  
**Why:** Initial weights influence how well signals and gradients flow at the start of training.  
**Expected output:** Five histories, convergence plot, zeros-failure plot and before/after weight histograms.

In [ ]:
initializers = ["glorot_uniform", "glorot_normal", "he_uniform", "he_normal", "zeros"]
initializer_models, initializer_histories, initializer_metrics = {}, {}, {}
initial_weight_snapshots = {}
for init_name in initializers:
    print("Training initializer:", init_name)
    model = build_ann(input_dim, activation="relu", initializer=init_name)
    initial_weight_snapshots[init_name] = model.layers[1].get_weights()[0].copy()
    hist = fit_model(model, epochs=50)
    metrics, _ = evaluate_model(model, verbose=False)
    initializer_models[init_name] = model; initializer_histories[init_name] = hist
    initializer_metrics[init_name] = metrics
    add_result(f"Initializer - {init_name}", model, hist, initializer=init_name)
    if init_name in ["he_normal", "glorot_uniform"]:
        initial_weight_snapshots[init_name + "_after"] = model.layers[1].get_weights()[0].copy()

### 7.2 Convergence speed comparison
Plot validation accuracy for all five initializers. The 84% and 82% lines are assignment reference thresholds, not guaranteed outcomes. Vertical markers show the first epoch crossing 82%, where a crossing occurs.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
styles = ["-", "--", "-.", ":", (0, (3, 1, 1, 1))]
for (name, hist), style in zip(initializer_histories.items(), styles):
    vals = hist.history["val_accuracy"]
    ax.plot(vals, linestyle=style, linewidth=2, label=name)
    crossed = np.where(np.asarray(vals) >= 0.82)[0]
    if name != "zeros" and len(crossed):
        epoch = int(crossed[0]); ax.axvline(epoch, alpha=0.22, linestyle=style)
        ax.annotate(f"{name}: {epoch+1}", (epoch, vals[epoch]), fontsize=8)
ax.axhline(0.84, color="black", linestyle="--", label="84% target")
ax.set(title="Weight Initialisation — Convergence Speed Comparison (Adult Income Dataset)", xlabel="Epoch", ylabel="Validation accuracy")
ax.legend(ncol=2)
save_figure("initialiser_convergence.png")

### 7.3 Zeros initialization failure
**Why:** With identical zero weights, hidden neurons remain symmetric and cannot learn distinct feature patterns. Expect this model to perform poorly; the exact curve is measured, not hard-coded. The assignment describes a near-majority baseline result, but the actual value must come from this run.

In [ ]:
zero_hist = initializer_histories["zeros"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(zero_hist.history["loss"], label="Training loss"); axes[0].plot(zero_hist.history["val_loss"], label="Validation loss")
axes[0].set(title="Zeros Initialization — Loss", xlabel="Epoch", ylabel="Loss"); axes[0].legend()
axes[1].plot(zero_hist.history["val_accuracy"], color="#CC0000", label="Validation accuracy")
axes[1].axhline(float(y_val.mean()), color="gray", linestyle="--", label="Positive-class prevalence")
axes[1].set(title="Zeros Initialization — Validation Accuracy", xlabel="Epoch", ylabel="Accuracy"); axes[1].legend()
save_figure("zeros_initialization_failure.png")
print("Zeros initializer test metrics:", initializer_metrics["zeros"])

### 7.4 Initial vs trained weight distributions
**What:** Compare first-layer weights before and after training for He Normal and Glorot Uniform.  
**Why:** The initializer sets starting values; backpropagation subsequently updates the distribution.  
**Expected output:** Four histograms in a 2×2 grid.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, key, title in zip(axes.flat,
    ["he_normal", "he_normal_after", "glorot_uniform", "glorot_uniform_after"],
    ["He Normal — Before", "He Normal — After", "Glorot Uniform — Before", "Glorot Uniform — After"]):
    ax.hist(initial_weight_snapshots[key].ravel(), bins=40, edgecolor="black", alpha=0.8)
    ax.set(title=title, xlabel="Weight value", ylabel="Frequency")
fig.suptitle("Weight Distributions — Initial vs Trained")
save_figure("weight_distributions.png")

## 8. Task 5 — Loss Functions
### 8.1 Binary Cross-Entropy (BCE)
For a true label y and predicted probability p, binary cross-entropy is **L = −[y log(p) + (1−y) log(1−p)]**. It is the natural loss for Bernoulli/binary classification and strongly penalizes confident incorrect predictions. With sigmoid output, the logit gradient simplifies to p−y.

### 8.2 MSE as a classification loss
MSE treats the probability like a regression target. Combined with sigmoid, its gradient includes the sigmoid derivative p(1−p), which becomes small at saturation. We compare its validation F1 with BCE.

In [ ]:
model_mse = build_ann(input_dim, activation="relu", initializer="he_normal", loss="mean_squared_error")
history_mse = fit_model(model_mse, epochs=50)
mse_metrics, _ = add_result("MSE", model_mse, history_mse, initializer="he_normal", loss_name="mean_squared_error")
# Compute validation F1 at each epoch from validation probabilities.
def epoch_f1_curve(model, history, X_valid=X_val, y_valid=y_val):
    # Use final model predictions as a transparent comparison; epoch-wise curves are collected below in a callback.
    return f1_score(y_valid, (model.predict(X_valid, verbose=0).ravel() >= 0.5).astype(int), zero_division=0)
print("MSE test metrics:", mse_metrics)
print("Note: epoch-wise F1 curves are calculated with the callback implementation in the next cell for BCE and MSE comparison.")

### 8.3 Validation F1 callback and BCE vs MSE experiment
A callback calculates class-1 F1 after each epoch on the fixed validation set. It does not update model weights; it only reports a validation metric.

In [ ]:
class ValidationF1(keras.callbacks.Callback):
    def __init__(self, x_val, y_val):
        super().__init__(); self.x_val=x_val; self.y_val=np.asarray(y_val).astype(int); self.values=[]
    def on_epoch_end(self, epoch, logs=None):
        p=self.model.predict(self.x_val, verbose=0).ravel()
        self.values.append(f1_score(self.y_val, (p>=0.5).astype(int), zero_division=0))

# Refit comparable models with the F1 callback so that the requested curves are measured per epoch.
model_bce_f1 = build_ann(input_dim, activation="relu", initializer="he_normal")
cb_bce = ValidationF1(X_val, y_val); hist_bce_f1 = model_bce_f1.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=50,batch_size=BATCH_SIZE,callbacks=[cb_bce],verbose=0)
model_mse_f1 = build_ann(input_dim, activation="relu", initializer="he_normal", loss="mean_squared_error")
cb_mse = ValidationF1(X_val, y_val); hist_mse_f1 = model_mse_f1.fit(X_train,y_train,validation_data=(X_val,y_val),epochs=50,batch_size=BATCH_SIZE,callbacks=[cb_mse],verbose=0)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(cb_bce.values, label="BCE"); ax.plot(cb_mse.values, label="MSE")
ax.set(title="BCE vs MSE — Validation F1 for Class 1", xlabel="Epoch", ylabel="Validation F1 (class 1)"); ax.legend()
save_figure("loss_f1_comparison.png")

### 8.4 Weighted BCE — class imbalance
**What:** Compute balanced class weights from the training labels and pass them to `model.fit`.  
**Why:** Errors on the minority class receive greater training weight. Recall may improve, sometimes with a precision trade-off.  
**Expected output:** Precision, recall and F1 comparison for baseline and weighted BCE.

In [ ]:
class_values = np.array([0, 1])
class_weights = compute_class_weight(class_weight="balanced", classes=class_values, y=np.asarray(y_train).astype(int))
class_weight_dict = {int(k): float(v) for k, v in zip(class_values, class_weights)}
print("Class weights:", class_weight_dict)
model_wbce = build_ann(input_dim, activation="relu", initializer="he_normal")
history_wbce = fit_model(model_wbce, epochs=50, class_weight=class_weight_dict)
weighted_metrics, weighted_probs = add_result("Weighted BCE", model_wbce, history_wbce,
    initializer="he_normal", loss_name="weighted_binary_crossentropy")
comparison = pd.DataFrame([baseline_metrics, weighted_metrics], index=["Baseline BCE", "Weighted BCE"])
display(comparison[["Prec(1)","Recall(1)","F1(1)"]].round(4))
fig, ax = plt.subplots(figsize=(8, 5))
comparison[["Prec(1)","Recall(1)","F1(1)"]].plot(kind="bar", ax=ax)
ax.set(title="Baseline vs Weighted BCE — Class 1 Metrics", xlabel="Model", ylabel="Score", ylim=(0,1)); ax.legend(title="Metric")
save_figure("weighted_bce_metrics.png")

### 8.5 Focal Loss — focus on hard examples
Focal Loss multiplies cross-entropy by a modulating factor so that easy examples receive less weight. The assignment's α and γ values are used here. The implementation clips probabilities for numerical stability.

In [ ]:
def focal_loss(gamma=2.0, alpha=0.25):
    def loss(y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, (-1, 1)), tf.float32)
        y_pred = tf.clip_by_value(tf.cast(y_pred, tf.float32), 1e-7, 1.0-1e-7)
        bce = -y_true*tf.math.log(y_pred) - (1-y_true)*tf.math.log(1-y_pred)
        weight = alpha*y_true*tf.pow(1-y_pred,gamma) + (1-alpha)*(1-y_true)*tf.pow(y_pred,gamma)
        return tf.reduce_mean(weight*bce)
    return loss

model_focal = build_ann(input_dim, activation="relu", initializer="he_normal", loss=focal_loss(gamma=2.0, alpha=0.25))
history_focal = fit_model(model_focal, epochs=50)
focal_metrics, focal_probs = add_result("Focal Loss", model_focal, history_focal,
    initializer="he_normal", loss_name="focal_loss")
loss_compare = pd.DataFrame([baseline_metrics, weighted_metrics, focal_metrics], index=["BCE", "Weighted BCE", "Focal Loss"])
display(loss_compare[["Prec(1)","Recall(1)","F1(1)"]].round(4))
print("Focal Loss reduces the influence of easy cases; compare the measured class-1 recall and F1 above.")

### 8.6 Loss function summary
The best production loss should be chosen using held-out performance and the cost of false positives vs false negatives, not from a theoretical claim alone. Weighted BCE or Focal Loss may help with imbalance, but the validation/test results decide.

In [ ]:
loss_summary = pd.DataFrame([
    ["BCE", "−[y log(p)+(1−y)log(1−p)]", "Binary classification", "No explicit weighting", baseline_metrics["F1(1)"], baseline_metrics["Recall(1)" ]],
    ["Weighted BCE", "BCE × class weight", "Imbalanced classification", "Yes, class weights", weighted_metrics["F1(1)"], weighted_metrics["Recall(1)"]],
    ["MSE", "(y−p)²", "Regression; comparison only", "No", mse_metrics["F1(1)"], mse_metrics["Recall(1)"]],
    ["Focal Loss", "α(1−pₜ)^γ × BCE", "Hard / imbalanced examples", "Yes, focal weighting", focal_metrics["F1(1)"], focal_metrics["Recall(1)"]],
], columns=["Loss","Simplified formula","Best for","Imbalance handling","Test F1(1)","Test Recall(1)"])
display(loss_summary.round(4))
best_loss_name = max({"BCE":baseline_metrics,"Weighted BCE":weighted_metrics,"MSE":mse_metrics,"Focal Loss":focal_metrics}, key=lambda k: {"BCE":baseline_metrics,"Weighted BCE":weighted_metrics,"MSE":mse_metrics,"Focal Loss":focal_metrics}[k]["F1(1)"])
print("Highest class-1 F1 among these loss experiments:", best_loss_name)

## 9. Task 6 — Batch Normalization
### 9.1 Why Batch Normalization?
During training, updates to one layer can change the distribution of inputs received by the next layer. Batch Normalization normalizes pre-activation values within a mini-batch, then learns scale γ and shift β: **BN(x) = γ((x−μB)/√(σ²B+ε)) + β**. During inference it uses moving averages of the batch mean and variance. γ and β are trainable; moving statistics are not.

**What:** Add BatchNorm between Dense and activation.  
**Why:** It can stabilize and speed training, although improvement is dataset-dependent.  
**Expected output:** Model summary with trainable/non-trainable parameters and training-dynamics comparison.

In [ ]:
model_bn = build_ann(input_dim, activation="relu", initializer="he_normal", use_batch_norm=True)
model_bn.summary()
history_bn = fit_model(model_bn, epochs=50)
bn_metrics, _ = add_result("BatchNorm", model_bn, history_bn, initializer="he_normal", batch_norm=True)
print("BatchNorm test metrics:", bn_metrics)

### 9.2 With vs without BatchNorm
Compare training and validation loss and accuracy for the baseline and BatchNorm model. A smaller train-validation gap or faster convergence is an observation to verify, not a guaranteed outcome.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for hist, label in [(history_base,"Baseline"),(history_bn,"BatchNorm")]:
    axes[0].plot(hist.history["val_loss"], label=f"{label} val")
    axes[0].plot(hist.history["loss"], linestyle="--", alpha=0.7, label=f"{label} train")
    axes[1].plot(hist.history["val_accuracy"], label=f"{label} val")
    axes[1].plot(hist.history["accuracy"], linestyle="--", alpha=0.7, label=f"{label} train")
axes[0].set(title="Loss", xlabel="Epoch", ylabel="Loss"); axes[0].legend(fontsize=8)
axes[1].set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy"); axes[1].legend(fontsize=8)
fig.suptitle("Batch Normalization vs Baseline — Training Dynamics (Adult Income)")
save_figure("batchnorm_dynamics.png")

### 9.3 BatchNorm position experiment
Compare Dense → ReLU → BatchNorm with Dense → BatchNorm → ReLU. The canonical order in this project is Dense (linear) → BatchNorm → Activation. The comparison is empirical; this notebook does not claim statistical significance from a single train/validation split.

In [ ]:
def build_bn_position_model(position="before"):
    model = keras.Sequential(name=f"batchnorm_{position}_activation")
    model.add(keras.layers.Input(shape=(input_dim,)))
    for units in (128,64):
        model.add(keras.layers.Dense(units, kernel_initializer="he_normal", use_bias=False))
        if position == "before":
            model.add(keras.layers.BatchNormalization()); model.add(keras.layers.Activation("relu"))
        else:
            model.add(keras.layers.Activation("relu")); model.add(keras.layers.BatchNormalization())
    model.add(keras.layers.Dense(1, activation="sigmoid"))
    model.compile(optimizer=keras.optimizers.Adam(), loss="binary_crossentropy", metrics=["accuracy"])
    return model

bn_position_models, bn_position_histories = {}, {}
for position in ["after", "before"]:
    m = build_bn_position_model(position)
    h = fit_model(m, epochs=30)
    bn_position_models[position] = m; bn_position_histories[position] = h
fig, ax = plt.subplots(figsize=(8,5))
for position,h in bn_position_histories.items(): ax.plot(h.history["val_accuracy"], label=f"BN {position} activation")
ax.set(title="BatchNorm Position — Validation Accuracy", xlabel="Epoch", ylabel="Validation accuracy"); ax.legend()
save_figure("batchnorm_position.png")
for position,m in bn_position_models.items():
    add_result(f"BatchNorm {position}",m,bn_position_histories[position],initializer="he_normal",batch_norm=True)

### 9.4 Inspect learned gamma and beta
**What:** Read the first BatchNormalization layer's learned parameters.  
**Why:** γ controls scale and β controls shift after normalization. Values near zero for γ indicate a channel's normalized signal is strongly down-scaled.  
**Expected output:** Two bar charts with 128 values each.

In [ ]:
first_bn = next(layer for layer in model_bn.layers if isinstance(layer, keras.layers.BatchNormalization))
gamma, beta, moving_mean, moving_variance = first_bn.get_weights()
fig, axes = plt.subplots(2, 1, figsize=(13, 7))
axes[0].bar(np.arange(len(gamma)), gamma); axes[0].set(title="BatchNorm Learned Gamma (Scale)", xlabel="Neuron", ylabel="Gamma")
axes[1].bar(np.arange(len(beta)), beta); axes[1].set(title="BatchNorm Learned Beta (Shift)", xlabel="Neuron", ylabel="Beta")
save_figure("batchnorm_gamma_beta.png")
print("Neurons with |gamma| < 0.1:", int(np.sum(np.abs(gamma)<0.1)), "of", len(gamma))

## 10. Task 7 — Optimizers: Convergence & Final Comparison
### 10.1 Optimizer theory
- **SGD:** One global learning rate; simple but can converge slowly.
- **SGD + Momentum:** Adds a velocity term to smooth updates and accelerate consistent directions.
- **RMSprop:** Scales updates using a moving average of squared gradients.
- **Adam:** Combines first-moment momentum (β₁) and second-moment scaling (β₂), with bias correction. This adaptive behaviour can be useful for sparse, high-dimensional one-hot features.

**What:** Compare five optimizer configurations while keeping architecture, activation, initializer, loss and BatchNorm fixed.  
**Why:** Controlled comparisons help attribute differences to the optimizer.  
**Expected output:** Validation-accuracy convergence plot and test metrics.

In [ ]:
optimizer_configs = {
    "SGD": keras.optimizers.SGD(learning_rate=0.01),
    "SGD + Momentum": keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    "RMSprop": keras.optimizers.RMSprop(learning_rate=0.001),
    "Adam": "adam",
    "Adam explicit": keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999),
}
optimizer_models, optimizer_histories, optimizer_metrics = {}, {}, {}
for opt_name, opt in optimizer_configs.items():
    print("Training optimizer:", opt_name)
    m = build_ann(input_dim, activation="relu", initializer="he_normal", use_batch_norm=True, optimizer=opt)
    h = fit_model(m, epochs=50)
    optimizer_models[opt_name] = m; optimizer_histories[opt_name] = h
    optimizer_metrics[opt_name], _ = add_result(f"Optimizer - {opt_name}",m,h,initializer="he_normal",batch_norm=True,optimizer=opt_name)

### 10.2 Optimizer convergence plot
Dashed horizontal lines show 80%, 83% and 86% reference levels. The first epoch crossing 83% is marked only if the observed validation curve reaches it.

In [ ]:
fig, ax = plt.subplots(figsize=(11,6))
for name,h in optimizer_histories.items():
    vals=np.asarray(h.history["val_accuracy"]); ax.plot(vals,label=name)
    crossed=np.where(vals>=0.83)[0]
    if len(crossed):
        e=int(crossed[0]); ax.annotate(f"{name}: {e+1}",(e,vals[e]),xytext=(5,8),textcoords="offset points",fontsize=8,arrowprops={"arrowstyle":"->","lw":0.7})
for level in [0.80,0.83,0.86]: ax.axhline(level,linestyle="--",alpha=0.55,label=f"{level:.0%} reference")
ax.set(title="Optimizer Comparison — Validation Accuracy over 50 Epochs (Adult Income ANN)",xlabel="Epoch",ylabel="Validation accuracy")
ax.legend(ncol=2,fontsize=8)
save_figure("optimiser_convergence.png")
optimizer_summary=pd.DataFrame(optimizer_metrics).T
print("Highest class-1 F1:",optimizer_summary["F1(1)"].idxmax())
fastest={name: (int(np.where(np.asarray(h.history["val_accuracy"])>=0.83)[0][0])+1 if np.any(np.asarray(h.history["val_accuracy"])>=0.83) else np.nan) for name,h in optimizer_histories.items()}
print("First epoch reaching 83% (NaN = not reached):",fastest)

### 10.3 Learning-rate sensitivity — SGD vs Adam
Train SGD at 0.0001, 0.001, 0.01 and 0.1, and Adam at 0.0001, 0.001 and 0.01 for 30 epochs. Large learning rates may diverge or oscillate; small rates may learn slowly. The plot shows actual validation loss.

In [ ]:
lr_histories = {"SGD":{}, "Adam":{}}
for opt_name, rates in [("SGD",[0.0001,0.001,0.01,0.1]),("Adam",[0.0001,0.001,0.01])]:
    for lr in rates:
        opt = keras.optimizers.SGD(learning_rate=lr) if opt_name=="SGD" else keras.optimizers.Adam(learning_rate=lr)
        m=build_ann(input_dim,activation="relu",initializer="he_normal",use_batch_norm=True,optimizer=opt)
        h=fit_model(m,epochs=30)
        lr_histories[opt_name][lr]=h
fig,axes=plt.subplots(1,2,figsize=(13,4))
for ax,(opt_name, histories) in zip(axes,lr_histories.items()):
    for lr,h in histories.items(): ax.plot(h.history["val_loss"],label=f"lr={lr}")
    ax.set(title=f"{opt_name} learning rates",xlabel="Epoch",ylabel="Validation loss"); ax.legend()
fig.suptitle("Learning Rate Sensitivity — SGD vs Adam")
save_figure("learning_rate_sensitivity.png")

### 10.4 Select final configuration and train combined ANN
The final model uses ReLU, He Normal, BatchNorm and Adam. Weighted BCE is selected only if it improves validation class-1 F1 compared with unweighted BCE; the choice is based on the validation set, not test-set tuning. The test set is used once for final reporting.

In [ ]:
# Validation-only selection between BCE and class-weighted BCE.
val_f1_bce = f1_score(y_val, (model_bce_f1.predict(X_val,verbose=0).ravel()>=0.5).astype(int), zero_division=0)
val_f1_wbce = f1_score(y_val, (model_wbce.predict(X_val,verbose=0).ravel()>=0.5).astype(int), zero_division=0)
use_class_weights = val_f1_wbce > val_f1_bce
final_loss_label = "weighted BCE" if use_class_weights else "BCE"
print(f"Validation F1 — BCE: {val_f1_bce:.4f}; Weighted BCE: {val_f1_wbce:.4f}")
print("Final loss selection:", final_loss_label)
model_final = build_ann(input_dim, hidden_units=(128,64), activation="relu", initializer="he_normal",
                        use_batch_norm=True, optimizer="adam", loss="binary_crossentropy")
history_final = fit_model(model_final, epochs=80, class_weight=class_weight_dict if use_class_weights else None)
final_metrics, final_probs = add_result("Final Combined",model_final,history_final,activation="relu",initializer="he_normal",
                                        loss_name=final_loss_label,batch_norm=True,optimizer="adam")
print("Final held-out test metrics:"); display(pd.Series(final_metrics).round(4))

### 10.5 Full results comparison table
The table includes the baseline, best activation, best initializer, weighted BCE, Focal Loss, BatchNorm and final combined model, plus supporting experiment rows. All values are measured on the same held-out test split. The highlighted cells identify the maximum class-1 F1 and ROC-AUC.

In [ ]:
# Select representative rows for the required summary.
representative_names = ["Baseline", f"Activation - {best_activation}",
                        f"Initializer - {max(initializer_metrics,key=lambda k:initializer_metrics[k]['F1(1)'])}",
                        "Weighted BCE", "Focal Loss", "BatchNorm", "Final Combined"]
results_df = pd.DataFrame(results)
summary_df = results_df[results_df["Model"].isin(representative_names)].drop_duplicates(subset=["Model"],keep="last")
metric_columns = ["Test Acc","Prec(1)","Recall(1)","F1(1)","ROC-AUC"]
display(summary_df[["Model","Activation","Init","Loss","BatchNorm","Optimizer"]+metric_columns]
        .style.highlight_max(subset=["F1(1)","ROC-AUC"],color="#ffcccc").format({c:"{:.4f}" for c in metric_columns}))
results_df.to_csv("model_results_all.csv",index=False)
summary_df.to_csv("model_results_summary.csv",index=False)
# Save a plot-friendly image of the representative metrics table.
fig,ax=plt.subplots(figsize=(14,max(3,0.55*len(summary_df))))
ax.axis("off")
table_data=summary_df[["Model","Test Acc","Prec(1)","Recall(1)","F1(1)","ROC-AUC"]].round(3)
tab=ax.table(cellText=table_data.values,colLabels=table_data.columns,loc="center",cellLoc="center")
tab.auto_set_font_size(False);tab.set_fontsize(8);tab.scale(1,1.5)
ax.set_title("Adult Income — Model Results Summary",pad=18)
save_figure("results_table.png")

### 10.6 ROC curve comparison
**What:** Plot true-positive rate against false-positive rate at different thresholds.  
**Why:** ROC-AUC summarizes ranking ability across thresholds and complements the class-1 F1 score at threshold 0.5.  
**Expected output:** Baseline, Weighted BCE and Final Combined ROC curves with measured AUC in the legend.

In [ ]:
fig,ax=plt.subplots(figsize=(7,6))
for name,probs in [("Baseline",baseline_probs),("Weighted BCE",weighted_probs),("Final Combined",final_probs)]:
    fpr,tpr,_=roc_curve(y_test,probs); auc_value=roc_auc_score(y_test,probs)
    ax.plot(fpr,tpr,label=f"{name} (AUC={auc_value:.3f})")
ax.plot([0,1],[0,1],linestyle="--",color="gray",label="Random classifier")
ax.set(title="ROC Curves — Adult Income ANN",xlabel="False Positive Rate",ylabel="True Positive Rate")
ax.legend()
save_figure("roc_curves.png")

## 11. Conclusion
This project compares MLP design choices for Adult Income classification. Use the generated result table and plots to state which activation, initializer, loss, BatchNorm arrangement and optimizer performed best on the held-out data. Do not claim that a technique improved performance unless its measured metric supports that conclusion.

### Final interpretation prompts
1. Which model achieved the highest class-1 F1 and ROC-AUC?
2. Did weighted BCE or Focal Loss improve recall, and what happened to precision?
3. Did He initialization converge faster than Glorot in this run?
4. Did BatchNorm improve validation stability or test performance?
5. Which optimizer reached the 83% validation reference first (if any)?

**Limitations:** The Adult dataset reflects historical US census data and should not be treated as a fair or current proxy for individual income. Demographic attributes can encode social bias. This model is an educational exercise, not a tool for employment, lending or eligibility decisions.

## 12. Export and Submission Checklist
After all cells finish successfully:

```bash
jupyter nbconvert --to html DL_PR2.ipynb --output DL_PR2.html
```

Check that the `plots/` folder contains the generated figures. Save the notebook after execution so it includes outputs. Restart the kernel and run all cells from top to bottom before submitting. The notebook requires the original `adult.csv`; results are intentionally not pre-filled or fabricated.